In [11]:
import numpy as np
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# 1. Download and load the Fashion MNIST dataset
print('Downloading Fashion MNIST dataset (this may take 30-60 seconds)...')
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False)
X = fashion_mnist.data
y = fashion_mnist.target.astype(int)

print(f"Total dataset size: {X.shape[0]} images, each with {X.shape[1]} pixels.")

# 2. Take a subset of 12,000 total samples for faster computation
X_subset, _, y_subset, _ = train_test_split(
    X, y,
    train_size=12000,
    stratify=y,
    random_state=42
)

# 3. Split the subset into training (10,000) and testing (2,000) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset,
    train_size=10000,
    stratify=y_subset,
    random_state=42
)

print(f"Training Images: {X_train.shape[0]}")
print(f"Testing Images: {X_test.shape[0]}")

# 4. Feature Scaling (Normalizing pixel values from [0, 255] to [0.0, 1.0])
print(f"Before scaling -> Min: {X_train.min()}, Max: {X_train.max()}")

X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"After scaling -> Min: {X_train.min()}, Max: {X_train.max()}")

# 5. Evaluate KNN across multiple K values
k_values = [1, 3, 5, 7, 9, 15]
results = {}

print("\n" + "=" * 55)
print(f"{'K Value':<8} | {'Accuracy':<10} | {'Prediction Time (seconds)':<25}")
print("-" * 55)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    knn.fit(X_train, y_train)
    
    start_time = time.time()
    y_pred = knn.predict(X_test)
    elapsed_time = time.time() - start_time
    
    acc = accuracy_score(y_test, y_pred)
    results[k] = {
        "accuracy": acc,
        "time": elapsed_time,
        "predictions": y_pred
    }
    
    print(f"{k:<8} | {acc * 100:<9.2f}% | {elapsed_time:<25.2f}")

print("=" * 55)

# 6. Find the best K and print the classification report
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

best_k = max(results, key=lambda k: results[k]["accuracy"])
print(f"\nBest K is: {best_k} with {results[best_k]['accuracy'] * 100:.2f}% accuracy\n")

print("Per-class classification report:")
print(classification_report(y_test, results[best_k]["predictions"], target_names=class_names))

Total dataset size: 70000 images, each with 784 pixels.
Training Images: 10000
Testing Images: 2000
Before scaling -> Min: 0, Max: 255
After scaling -> Min: 0.0, Max: 1.0

K Value  | Accuracy   | Prediction Time (seconds)
-------------------------------------------------------
1        | 79.30    % | 1.69                     
3        | 80.90    % | 0.13                     
5        | 81.00    % | 0.13                     
7        | 81.10    % | 0.14                     
9        | 80.95    % | 0.14                     
15       | 80.25    % | 0.14                     

Best K is: 7 with 81.10% accuracy

Per-class classification report:
              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00     